In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import pandas as pd

load_dotenv()

True

In [2]:
# OpenAI:
# llm = init_chat_model("openai:gpt-3.5-turbo")

# embedding_model = "text-embedding-3-small" # outputs 1536 dimensions by default
# embedding = OpenAIEmbeddings(model=embedding_model)

# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="qwen3-embedding:8b")

In [3]:

fnx_doc = []

try:
    policies_table = pd.read_excel("../DataIngestParsing/data/csv_excel/policies_table.xlsx")
    users_table = pd.read_excel("../DataIngestParsing/data/csv_excel/users_table.xlsx")

    for _, row in policies_table.iterrows():
        matched = users_table[users_table["Id"] == row["מזהה לקוח"]]
        user_row = matched.iloc[0] if not matched.empty else pd.Series(dtype=object)

        text = f"""
        פוליסה מספר {row.get('מספר הפוליסה', '')},

        בעל הפוליסה: {user_row.get('שם פרטי', '')} {user_row.get('שם משפחה', '')}
        תעודת זהות: {user_row.get('תעודת זהות', '')}

        כתובת בעל הפוליסה:
        {user_row.get('עיר', '')}, {user_row.get('רחוב', '')} {user_row.get('מספר בית', '')}

        תאריך לידה: {user_row.get('תאריך לידה', '')}
        שנת הוצאת רישיון: {user_row.get('שנת הוצאת רישיון', '')}
        כתובת אימייל: {user_row.get('Email', '')}

        הפוליסה מנוהלת על ידי הסוכן: {row.get('שם סוכן', '')}
        הפוליסה נוצרה על ידי: {row.get('נציג', '')}
        הפוליסה עודכנה לאחרונה על ידי: {row.get('עודכן ע"י', '')}

        סטטוס הפוליסה: {"לא פעילה" if row.get("IsActive") == 0 else "פעילה"}.
        """

        # Clean metadata
        metadata = {}

        for key, value in row.items():

            # Skip NaN
            if pd.isna(value):
                continue

            # Convert timestamps to string
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()

            # Convert numpy types to native Python
            if hasattr(value, "item"):
                value = value.item()

            metadata[str(key)] = value

        fnx_doc.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

    print(f"The embedding data will look like that: {fnx_doc[0]}")
except Exception as e:
    print(f"Error loading PyPDF: {e}")


The embedding data will look like that: page_content='
        פוליסה מספר 1136442,

        בעל הפוליסה: ישי אמסלם
        תעודת זהות: 325695120

        כתובת בעל הפוליסה:
        מודיעין-מכבים-רעו, עפרוני 32

        תאריך לידה: 2003-12-20 00:00:00
        שנת הוצאת רישיון: 2022
        כתובת אימייל: yishayamsa@gmail.com

        הפוליסה מנוהלת על ידי הסוכן: SMART הפניקס
        הפוליסה נוצרה על ידי: ישי אמסלם
        הפוליסה עודכנה לאחרונה על ידי: שלו קריו

        סטטוס הפוליסה: פעילה.
        ' metadata={'Id': 29130, 'PolicyId': 250961121136442, 'שנת חיתום': 25, 'BranchNumber': 96, 'Anaf': 112, 'מספר הפוליסה': 1136442, 'מזהה לקוח': 28871, 'תאריך תחילת הפוליסה': '2025-05-17T00:00:00', 'תאריך סיום הפוליסה': '2026-05-11T14:42:00.787000', 'סטטוס הפוליסה': 2, 'StatusDate': '2026-05-11T14:42:00.787000', 'מספר סוכן': 64096, 'שם סוכן': 'SMART הפניקס', 'InstanceId': 7682268845, 'ExternalStatus': 2.0, 'תאריך יצירת הרשומה': '2025-05-17T22:12:56.713000', 'נציג': 'ישי אמסלם', 'CreatorType': 1

In [4]:
 # Vector store: ChromaDB (Chroma.from_documents) automatically infers the dimension from the first embedding it receives — you don't need to configure it manually. It adapts to whatever model you pass.

persist_directory = "./chroma_db_store"
vector_store = Chroma.from_documents(
    documents=fnx_doc,
    embedding=embedding,
    persist_directory=persist_directory, # This will create chroma_db_store folder for the vector store with all the data
    collection_name="core"
)

print(f"Created vector store with {len(vector_store)} vectors")

Created vector store with 2346 vectors


In [13]:
# Similarity TEST: Similarity search works on page_content, NOT metadata
query = "ישי אמסלם"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = vector_store.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores)

[(Document(metadata={'NumberOfClaims': 0.0, 'InstanceId': 5921575285, 'תאריך תחילת הפוליסה': '2026-05-03T00:00:00', 'תאריך סיום הפוליסה': '2027-04-30T23:59:59', 'IsAgent': 1, 'מזהה לקוח': 28093, 'שנת חיתום': 26, 'IsActive': 1, 'ExternalStatus': 2.0, 'תאריך יצירת הרשומה': '2026-05-03T19:05:29.057000', 'Id': 87955, 'מספר סוכן': 47549, 'NumberOfDenials': 0.0, 'PrevPolicyId': 28335.0, 'CreatorType': 1, 'נציג': 'תמר  אמסלם ', 'שם סוכן': 'יעקב מנצר ביטוח בע"מ', 'StatusDate': '2026-05-03T19:05:29.057000', 'PolicyId': 260361121001108, 'Anaf': 112, 'מספר הפוליסה': 1001108, 'סטטוס הפוליסה': 1, 'BranchNumber': 36}, page_content='\n        פוליסה מספר 1001108,\n\n        בעל הפוליסה: תמר  אמסלם \n        תעודת זהות: 217579697\n\n        כתובת בעל הפוליסה:\n        ירושלים, השי ש 1\n\n        תאריך לידה: 2008-08-27 00:00:00\n        שנת הוצאת רישיון: 2025\n        כתובת אימייל: Tmrmslm25@gmail.com\n\n        הפוליסה מנוהלת על ידי הסוכן: יעקב מנצר ביטוח בע"מ\n        הפוליסה נוצרה על ידי: תמר  אמסלם

In [6]:
custom_prompt = """
        You are a professional customer service representative for Phoenix Insurance Company.
        ***Always respond in Hebrew regardless of the language used in the question***.

        Your goal is to provide accurate information about the company's policies.

        Guidelines:
        * Use a professional, patient, and respectful tone.
        * If asked analytical questions, respond that you are not an analytics tool — DO NOT RETURN ANY DATA!
        * If information is missing or not found in the system — state it clearly.
        * Do not fabricate information that does not exist in the provided data.
        * If multiple similar results are found — ask for additional details to identify the correct one.
        * If the question is unrelated to Phoenix or the data in the system — politely respond that you cannot assist with that topic.
        * Phrase answers in a natural, human way — not as a raw technical data list.

        Example response style:
        * "Your policy is active until 11/05/2026..."
        * "The agent handling your policy is SMART הפניקס..."
        * "No policy was found matching the provided details..."

        ***Always respond in Hebrew***.

        Context: {context}
    """

In [7]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
import re
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field
from typing import Any

# BM25 retriever (Exact mach for names - names are not embed great so the semantic search is weak)
bm25_retriever = BM25Retriever.from_documents(fnx_doc, k=3)
# Dense retriever - Regular similarity search
semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 80/50 blend — tune weights based on your results
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.8, 0.2]
)

class HybridRetriever(BaseRetriever):
    vector_store: Any = Field(...)
    ensemble: Any = Field(...)
    k: int = Field(default=3)

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # 1. Exact policy number lookup via metadata
        policy_match = re.search(r'\b(\d{6,10})\b', query)
        if policy_match:
            policy_number = int(policy_match.group(1))
            results = self.vector_store.get(where={"הסילופה רפסמ": policy_number})
            docs = [
                Document(page_content=pc, metadata=meta)
                for pc, meta in zip(results["documents"], results["metadatas"])
            ]
            if docs:
                return docs

        # 2. BM25 + semantic ensemble for everything else (names, questions, etc.)
        return self.ensemble.invoke(query)

retriever = HybridRetriever(vector_store=vector_store, ensemble=ensemble_retriever)

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", custom_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        You are a professional customer service representative for Phoenix Insurance Company.\n        ***Always respond in Hebrew regardless of the language used in the question***.\n\n        Your goal is to provide accurate information about the company\'s policies.\n\n        Guidelines:\n        * Use a professional, patient, and respectful tone.\n        * If asked analytical questions, respond that you are not an analytics tool — DO NOT RETURN ANY DATA!\n        * If information is missing or not found in the system — state it clearly.\n        * Do not fabricate information that does not exist in the provided data.\n        * If multiple similar results are found — ask for additional details to identify the correct one.\n        * If the question i

In [9]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [12]:
# response = rag_chain.invoke({ "input": "האם קיים לקוח בשם יונתן מורל?" })
# response = rag_chain.invoke({ "input": "האם לאגם וקנין יש פוליסה פעילה?" })
# response = rag_chain.invoke({ "input": "מי בעל הפוליסה 1015082?" })
response = rag_chain.invoke({ "input": "מה המייל של ינון אליה דוידי?" })
# response = rag_chain.invoke({ "input": "מה הכתובת של בעל הפוליסה 1029855?" })

print(response.get("answer"))

שלום, המייל של ינון אליה דוידי הוא inon211280@gmail.com.
